In [1]:
!pip install geopandas folium shapely pandas

In [2]:
import pandas as pd
import geopandas as gpd
import folium
from shapely.geometry import Point
from folium.plugins import MarkerCluster
from google.colab import files

In [3]:
data = {
    "asset_id": [f"AST-{i:03d}" for i in range(1, 11)],
    "asset_name": [
        "Pohon Mahoni Taman Kota", "Pohon Trembesi Alun-alun", "Pohon Angsana Jl. Merdeka",
        "Lampu Jalan Sub-01", "Pohon Beringin Hutan Kota", "Lampu Taman Sektor A",
        "Pohon Palem Pagar Alam", "Lampu Jalan Sub-02", "Pohon Akasia Jl. Sudirman", "Lampu Taman Sektor B"
    ],
    "category": [
        "Pohon", "Pohon", "Pohon", "Penerangan", "Pohon",
        "Penerangan", "Pohon", "Penerangan", "Pohon", "Penerangan"
    ],
    "condition": [
        "Rawat Segera", "Baik", "Perlu Pemangkasan", "Rusak", "Baik",
        "Baik", "Rawat Segera", "Baik", "Perlu Pemangkasan", "Rusak"
    ],
    "latitude": [-4.3210, -4.3225, -4.3205, -4.3240, -4.3190, -4.3218, -4.3250, -4.3233, -4.3201, -4.3261],
    "longitude": [103.2500, 103.2520, 103.2485, 103.2510, 103.2530, 103.2490, 103.2540, 103.2505, 103.2478, 103.2525]
}

df_assets = pd.DataFrame(data)
df_assets.to_csv("sample_assets.csv", index=False)
print("File 'sample_assets.csv' berhasil dibuat!")
df_assets.head()

File 'sample_assets.csv' berhasil dibuat!


,asset_id,asset_name,category,condition,latitude,longitude
0,AST-001,Pohon Mahoni Taman Kota,Pohon,Rawat Segera,-4.3210,103.2500
1,AST-002,Pohon Trembesi Alun-alun,Pohon,Baik,-4.3225,103.2520
2,AST-003,Pohon Angsana Jl. Merdeka,Pohon,Perlu Pemangkasan,-4.3205,103.2485
3,AST-004,Lampu Jalan Sub-01,Penerangan,Rusak,-4.3240,103.2510
4,AST-005,Pohon Beringin Hutan Kota,Pohon,Baik,-4.3190,103.2530


In [4]:
df = pd.read_csv("sample_assets.csv")

geometry = [Point(xy) for xy in zip(df['longitude'], df['latitude'])]
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

print("Data berhasil diubah menjadi GeoDataFrame Spasial:")
gdf.head()

Data berhasil diubah menjadi GeoDataFrame Spasial:


,asset_id,asset_name,category,condition,latitude,longitude,geometry
0,AST-001,Pohon Mahoni Taman Kota,Pohon,Rawat Segera,-4.3210,103.2500,POINT (103.25 -4.321)
1,AST-002,Pohon Trembesi Alun-alun,Pohon,Baik,-4.3225,103.2520,POINT (103.252 -4.3225)
2,AST-003,Pohon Angsana Jl. Merdeka,Pohon,Perlu Pemangkasan,-4.3205,103.2485,POINT (103.2485 -4.3205)
3,AST-004,Lampu Jalan Sub-01,Penerangan,Rusak,-4.3240,103.2510,POINT (103.251 -4.324)
4,AST-005,Pohon Beringin Hutan Kota,Pohon,Baik,-4.3190,103.2530,POINT (103.253 -4.319)


In [5]:
priority_assets = gdf[gdf['condition'].isin(["Rawat Segera", "Rusak", "Perlu Pemangkasan"])]

print(f"Total Seluruh Aset: {len(gdf)}")
print(f"Aset Membutuhkan Perawatan/Perbaikan: {len(priority_assets)}")

priority_assets[['asset_id', 'asset_name', 'category', 'condition']]

Total Seluruh Aset: 10
Aset Membutuhkan Perawatan/Perbaikan: 6


,asset_id,asset_name,category,condition
0,AST-001,Pohon Mahoni Taman Kota,Pohon,Rawat Segera
2,AST-003,Pohon Angsana Jl. Merdeka,Pohon,Perlu Pemangkasan
3,AST-004,Lampu Jalan Sub-01,Penerangan,Rusak
6,AST-007,Pohon Palem Pagar Alam,Pohon,Rawat Segera
8,AST-009,Pohon Akasia Jl. Sudirman,Pohon,Perlu Pemangkasan
9,AST-010,Lampu Taman Sektor B,Penerangan,Rusak


In [10]:
center_lat = gdf['latitude'].mean()
center_lon = gdf['longitude'].mean()

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=15,
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles &copy; Esri &mdash; Source: Esri, DeLorme, NAVTEQ, USGS, Intermap, iPC, NRCAN, Esri Japan, METI, Esri China, TomTom"
)

marker_cluster = MarkerCluster(name="Semua Aset").add_to(m)

color_map = {
    "Baik": "green",
    "Perlu Pemangkasan": "orange",
    "Rawat Segera": "red",
    "Rusak": "darkred"
}

for idx, row in gdf.iterrows():
    color = color_map.get(row['condition'], 'blue')

    popup_html = f"""
    <div style="font-family: Arial; width: 200px;">
        <h4><b>{row['asset_name']}</b></h4>
        <b>ID Aset:</b> {row['asset_id']}<br>
        <b>Kategori:</b> {row['category']}<br>
        <b>Kondisi:</b> <span style="color:{color}; font-weight:bold;">{row['condition']}</span><br>
        <b>Koordinat:</b> {row['latitude']:.4f}, {row['longitude']:.4f}
    </div>
    """

    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=f"{row['asset_name']} ({row['condition']})",
        icon=folium.Icon(color=color, icon="info-sign" if row['category'] == "Penerangan" else "tree-conifer")
    ).add_to(marker_cluster)

m.save("index.html")
print("Peta interaktif berhasil diperbarui dan disimpan sebagai 'index.html'")

Peta interaktif berhasil diperbarui dan disimpan sebagai 'index.html'


In [7]:
priority_assets.to_file("priority_assets.geojson", driver="GeoJSON")

print("Mengunduh file 'index.html' dan 'priority_assets.geojson'...")

files.download("index.html")
files.download("priority_assets.geojson")

Mengunduh file 'index.html' dan 'priority_assets.geojson'...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>